# Unsupervised Pair Ranking


In [6]:
import re
import numpy as np
from pathlib import Path
from collections import defaultdict

TXT_PATH = Path("/home/ac.zzheng/power/GPGPU/data/H100/h100_rank_results.txt")

FEATURE_KEYS = ["dram_min_scaled", "sm_min_scaled", "fp_min_scaled"]

# Global/default seeds
SEED_W_SPEC = np.array([0.75, 0.15, 0.10], dtype=float)
SEED_W_ML   = np.array([0.75, 0.15, 0.10], dtype=float)

# Profiling-detected overhead-dominated regime seed
SEED_W_OVERHEAD = np.array([1.2, -0.6, -1.0], dtype=float)

SEED_B = 0.0

assert len(SEED_W_SPEC) == len(FEATURE_KEYS)
assert len(SEED_W_ML) == len(FEATURE_KEYS)
assert len(SEED_W_OVERHEAD) == len(FEATURE_KEYS)

LR = 0.03
EPOCHS = 800
L2_SEED = 0.10
TIE_THRESH = 0.03

# Interpretable profiling-only gate thresholds
GATE_CFG = {
    "rho_sm_min": 0.8,
    "rho_dram_min": 0.5,
    "fp_flat_max": 1.25,
    "sm_spread_min": 2.0,
}


def sigmoid(z):
    z = np.clip(z, -40, 40)
    return 1.0 / (1.0 + np.exp(-z))


def parse_txt_9col(path):
    lines = path.read_text().splitlines()
    rows = []
    suite = app = better = None
    cap = None
    in_table = False

    header = (
        "gpu_count	performance	dram_sum	sm_sum	fp_sum	"
        "dram_min_scaled	sm_min_scaled	fp_min_scaled"
    )

    for ln in lines:
        if ln.startswith("===== ") and ln.endswith(" ====="):
            m = re.match(r"^=====\s+(\w+)/(.*?)\s+=====$", ln)
            if m:
                suite = m.group(1).strip().upper()
                app = m.group(2).strip()
            continue

        if ln.startswith("(performance better when "):
            better = "lower" if "lower" in ln else "higher"
            continue

        m = re.match(r"^cap=(\d+)\s+true_rank=\[(.*?)\]$", ln)
        if m:
            cap = int(m.group(1))
            in_table = False
            continue

        if ln.startswith(header):
            in_table = True
            continue

        if in_table and ln.strip():
            ps = ln.split("	")
            if len(ps) == 8 and ps[0].isdigit():
                rows.append({
                    "suite": suite,
                    "app": app,
                    "cap": cap,
                    "gpu": int(ps[0]),
                    "perf": float(ps[1]),
                    "dram_sum": float(ps[2]),
                    "sm_sum": float(ps[3]),
                    "fp_sum": float(ps[4]),
                    "dram_min_scaled": float(ps[5]),
                    "sm_min_scaled": float(ps[6]),
                    "fp_min_scaled": float(ps[7]),
                    "better": better,
                })
            else:
                in_table = False

    by_case = defaultdict(list)
    for r in rows:
        by_case[(r["suite"], r["app"], r["cap"])].append(r)
    return by_case


def minmax_norm(vals):
    lo, hi = min(vals), max(vals)
    if hi - lo < 1e-12:
        return [0.0] * len(vals)
    return [(v - lo) / (hi - lo) for v in vals]


def build_case_feats(rows):
    feat_cols = [minmax_norm([r[k] for r in rows]) for k in FEATURE_KEYS]
    feats = []
    for i, r in enumerate(rows):
        vec = np.array([feat_cols[j][i] for j in range(len(FEATURE_KEYS))], dtype=float)
        feats.append({
            "gpu": r["gpu"],
            "perf": r["perf"],
            "feat": vec,
            "better": r["better"],
            "dram_min_scaled": r["dram_min_scaled"],
            "sm_min_scaled": r["sm_min_scaled"],
            "fp_min_scaled": r["fp_min_scaled"],
        })
    return feats


def rankdata_avg_tie(vals):
    vals = np.asarray(vals, dtype=float)
    order = np.argsort(vals)
    ranks = np.empty(len(vals), dtype=float)
    i = 0
    while i < len(vals):
        j = i
        while j + 1 < len(vals) and vals[order[j + 1]] == vals[order[i]]:
            j += 1
        avg_rank = (i + j) / 2.0 + 1.0
        ranks[order[i:j + 1]] = avg_rank
        i = j + 1
    return ranks


def spearman_corr(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if len(x) < 2:
        return 0.0
    if np.allclose(x, x[0]) or np.allclose(y, y[0]):
        return 0.0
    rx = rankdata_avg_tie(x)
    ry = rankdata_avg_tie(y)
    rx = rx - rx.mean()
    ry = ry - ry.mean()
    den = np.sqrt((rx * rx).sum() * (ry * ry).sum())
    if den <= 1e-12:
        return 0.0
    return float((rx * ry).sum() / den)


def safe_ratio_max_min(vals):
    vals = np.asarray(vals, dtype=float)
    if len(vals) == 0:
        return 1.0
    vmin = float(vals.min())
    vmax = float(vals.max())
    if vmin <= 1e-12:
        return float("inf") if vmax > 1e-12 else 1.0
    return vmax / vmin


def detect_overhead_regime(feats, cfg=GATE_CFG):
    g = np.array([x["gpu"] for x in feats], dtype=float)
    sm = np.array([x["sm_min_scaled"] for x in feats], dtype=float)
    dram = np.array([x["dram_min_scaled"] for x in feats], dtype=float)
    fp = np.array([x["fp_min_scaled"] for x in feats], dtype=float)

    rho_sm = spearman_corr(g, sm)
    rho_dram = spearman_corr(g, dram)
    fp_flat = safe_ratio_max_min(fp)
    sm_spread = safe_ratio_max_min(sm)

    is_overhead = (
        rho_sm >= cfg["rho_sm_min"] and
        rho_dram >= cfg["rho_dram_min"] and
        fp_flat <= cfg["fp_flat_max"] and
        sm_spread >= cfg["sm_spread_min"]
    )

    stats = {
        "rho_sm": rho_sm,
        "rho_dram": rho_dram,
        "fp_flat": fp_flat,
        "sm_spread": sm_spread,
    }
    return is_overhead, stats


def choose_seed_by_profile(feats, seed_default, seed_overhead):
    is_overhead, stats = detect_overhead_regime(feats)
    return (seed_overhead if is_overhead else seed_default), is_overhead, stats


def build_pseudo_pairs(feats, seed_w, seed_b):
    X, y = [], []
    seed_scores = [float(seed_w @ x["feat"] + seed_b) for x in feats]
    for i in range(len(feats)):
        for j in range(i + 1, len(feats)):
            diff = feats[i]["feat"] - feats[j]["feat"]
            yi = 1.0 if seed_scores[i] > seed_scores[j] else 0.0
            X.append(diff)
            y.append(yi)
            X.append(-diff)
            y.append(1.0 - yi)
    return np.array(X, dtype=float), np.array(y, dtype=float)


def train_case_unsup(X, y, w0, b0=0.0, lr=0.03, epochs=800, l2_seed=0.10):
    w = np.array(w0, dtype=float).copy()
    b = float(b0)
    n = len(y)
    if n == 0:
        return w, b
    for _ in range(epochs):
        p = sigmoid(X @ w + b)
        grad_w = (X.T @ (p - y)) / n + l2_seed * (w - w0)
        grad_b = np.mean(p - y)
        w -= lr * grad_w
        b -= lr * grad_b
    return w, b


def rank_case(feats, w, b):
    scores = np.zeros(len(feats), dtype=float)
    for i in range(len(feats)):
        for j in range(len(feats)):
            if i == j:
                continue
            scores[i] += sigmoid((feats[i]["feat"] - feats[j]["feat"]) @ w + b)
    order = np.argsort(-scores)
    return [feats[k]["gpu"] for k in order]


def true_rank(feats, better):
    if better == "lower":
        idx = sorted(range(len(feats)), key=lambda i: feats[i]["perf"])
    else:
        idx = sorted(range(len(feats)), key=lambda i: feats[i]["perf"], reverse=True)
    return [feats[i]["gpu"] for i in idx]


def top1_true_set(feats, better, tie_thresh=0.02):
    if not feats:
        return set()

    if better == "lower":
        ranked = sorted(feats, key=lambda x: x["perf"])
        best = ranked[0]["perf"]
        if len(ranked) == 1 or best <= 0:
            return {ranked[0]["gpu"]}
        second = ranked[1]["perf"]
        rel_gap = (second - best) / best
        return {ranked[0]["gpu"], ranked[1]["gpu"]} if rel_gap <= tie_thresh else {ranked[0]["gpu"]}

    ranked = sorted(feats, key=lambda x: x["perf"], reverse=True)
    best = ranked[0]["perf"]
    if len(ranked) == 1 or best <= 0:
        return {ranked[0]["gpu"]}
    second = ranked[1]["perf"]
    rel_gap = (best - second) / best
    return {ranked[0]["gpu"], ranked[1]["gpu"]} if rel_gap <= tie_thresh else {ranked[0]["gpu"]}


def run_eval(target_suite, by_case, seed_w_default, seed_w_overhead=None, use_profile_gate=False):
    cases = [(k, v) for k, v in sorted(by_case.items()) if k[0] == target_suite]
    print("\n{} cases:".format(target_suite), len(cases))
    print("FEATURE_KEYS:", FEATURE_KEYS)
    print("SEED_W_{}_DEFAULT:".format(target_suite), seed_w_default.tolist())
    if seed_w_overhead is not None:
        print("SEED_W_OVERHEAD:", seed_w_overhead.tolist())
    if use_profile_gate:
        print("PROFILE_GATE_CFG:", GATE_CFG)

    hits = 0
    results = []
    weights = []

    overhead_hits = 0
    overhead_total = 0

    for (suite, app, cap), rows in cases:
        feats = build_case_feats(rows)

        used_overhead = False
        gate_stats = None
        seed_w = np.array(seed_w_default, dtype=float)
        if use_profile_gate:
            seed_w, used_overhead, gate_stats = choose_seed_by_profile(
                feats,
                seed_default=np.array(seed_w_default, dtype=float),
                seed_overhead=np.array(seed_w_overhead, dtype=float),
            )
            overhead_total += int(used_overhead)

        X, y = build_pseudo_pairs(feats, seed_w, SEED_B)

        w_case, b_case = train_case_unsup(
            X, y, w0=seed_w, b0=SEED_B, lr=LR, epochs=EPOCHS, l2_seed=L2_SEED
        )

        pred = rank_case(feats, w_case, b_case)
        better = feats[0]["better"] if feats else "lower"
        trank = true_rank(feats, better)
        tset = top1_true_set(feats, better, tie_thresh=TIE_THRESH)
        hit = int(pred[0] in tset)

        if used_overhead:
            overhead_hits += hit

        hits += hit
        results.append((app, cap, pred, trank, sorted(tset), hit, used_overhead, gate_stats))
        weights.append(w_case)


    acc = hits / len(cases) if cases else 0.0
    print("\nTop-1 accuracy ({}): {:.4f} ({}/{})".format(target_suite, acc, hits, len(cases)))

    if use_profile_gate:
        print("Gate selected overhead seed on {}/{} cases".format(overhead_total, len(cases)))
        if overhead_total > 0:
            print("Top-1 on gated-overhead cases: {:.4f} ({}/{})".format(overhead_hits / overhead_total, overhead_hits, overhead_total))

    print("\nAverage learned per-case weights:")
    if len(weights) == 0:
        print("No parsed {} cases. Check txt format/header.".format(target_suite))
    else:
        W = np.vstack(weights)
        for i, k in enumerate(FEATURE_KEYS):
            print("w_{} =".format(k), round(float(W[:, i].mean()), 6))

    print("\nPred vs True (all cases):")
    for app, cap, pred, trank, tset, hit, used_overhead, gate_stats in results[0:0]:
        gate_tag = "OVERHEAD" if used_overhead else "DEFAULT"
        if use_profile_gate:
            print("{:<12s} cap={:4d} gate={:8s} pred_rank={} true_rank={}".format(app, cap, gate_tag, pred, trank))
        else:
            print("{:<12s} cap={:4d} pred_rank={} true_rank={}".format(app, cap, pred, trank))

    print("\nMispredictions:")
    for app, cap, pred, trank, tset, hit, used_overhead, gate_stats in results:
        if not hit:
            gate_tag = "OVERHEAD" if used_overhead else "DEFAULT"
            if use_profile_gate:
                print("{:<12s} cap={:4d} gate={:8s} pred_rank={} true_rank={}".format(app, cap, gate_tag, pred, trank))
            else:
                print("{:<12s} cap={:4d} pred_rank={} true_rank={}".format(app, cap, pred, trank))


by_case = parse_txt_9col(TXT_PATH)
print("Parsed total cases:", len(by_case))

# run_eval("SPEC", by_case, SEED_W_SPEC)
run_eval("ML", by_case, SEED_W_ML, seed_w_overhead=SEED_W_OVERHEAD, use_profile_gate=True)


Parsed total cases: 135

ML cases: 45
FEATURE_KEYS: ['dram_min_scaled', 'sm_min_scaled', 'fp_min_scaled']
SEED_W_ML_DEFAULT: [0.75, 0.15, 0.1]
SEED_W_OVERHEAD: [1.2, -0.6, -1.0]
PROFILE_GATE_CFG: {'rho_sm_min': 0.8, 'rho_dram_min': 0.5, 'fp_flat_max': 1.25, 'sm_spread_min': 2.0}

Top-1 accuracy (ML): 0.9111 (41/45)
Gate selected overhead seed on 1/45 cases
Top-1 on gated-overhead cases: 1.0000 (1/1)

Average learned per-case weights:
w_dram_min_scaled = 1.565018
w_sm_min_scaled = 0.255282
w_fp_min_scaled = 0.446264

Pred vs True (all cases):

Mispredictions:
resnet50     cap= 500 gate=DEFAULT  pred_rank=[1, 2] true_rank=[2, 1]
vgg16        cap= 800 gate=DEFAULT  pred_rank=[3, 1, 2, 4] true_rank=[1, 4, 3, 2]
vgg16        cap=1300 gate=DEFAULT  pred_rank=[2, 4, 3, 1] true_rank=[1, 3, 2, 4]
vgg16        cap=1400 gate=DEFAULT  pred_rank=[4, 1, 3, 2] true_rank=[1, 3, 2, 4]
